# Star schema: fact_spend with supplier, category, service area and date dimensions

## Source check

In [1]:
import os
import duckdb

con = duckdb.connect("../council.duckdb")
con.sql("SELECT COUNT(*) AS payments, SUM(amount) AS net_spend FROM clean_spend").show()

┌──────────┬───────────────┐
│ payments │   net_spend   │
│  int64   │ decimal(38,2) │
├──────────┼───────────────┤
│    77033 │  421596634.99 │
└──────────┴───────────────┘



## dim_category
15 LLM categories plus one for payments to individuals.

In [2]:
con.sql("""
CREATE OR REPLACE TABLE dim_category AS
WITH cats AS (
    SELECT DISTINCT category_llm_v1 AS category_name FROM supplier_llm_v1
    UNION
    SELECT 'Individuals (withheld)'
)
SELECT ROW_NUMBER() OVER (ORDER BY category_name) AS category_key, category_name
FROM cats
""")
con.sql("SELECT * FROM dim_category").show()

┌──────────────┬───────────────────────────┐
│ category_key │       category_name       │
│    int64     │          varchar          │
├──────────────┼───────────────────────────┤
│            1 │ Community & Voluntary     │
│            2 │ Construction & Property   │
│            3 │ Culture, Leisure & Events │
│            4 │ Education & Childcare     │
│            5 │ Energy & Utilities        │
│            6 │ Goods & Supplies          │
│            7 │ Housing & Accommodation   │
│            8 │ IT & Telecoms             │
│            9 │ Individuals (withheld)    │
│           10 │ Other                     │
│           11 │ Professional Services     │
│           12 │ Public Sector Bodies      │
│           13 │ Social Care               │
│           14 │ Staffing & Recruitment    │
│           15 │ Transport & Travel        │
│           16 │ Waste & Environment       │
└──────────────┴───────────────────────────┘
  16 rows                        2 columns



## dim_supplier
One row per standard supplier name (LLM v1). Redacted payments and suppliers the LLM flagged as private individuals are grouped as one withheld supplier (GDPR).

In [3]:
con.sql("""
CREATE OR REPLACE TABLE supplier_map AS
SELECT
    l.vendor_raw,
    CASE WHEN l.is_person_llm_v1 THEN 'Private individual (withheld)' ELSE l.std_name_llm_v1 END AS supplier_name,
    CASE WHEN l.is_person_llm_v1 THEN 'Individuals (withheld)' ELSE l.category_llm_v1 END AS category_name,
    l.is_person_llm_v1 AS is_individual,
    r.total_spend
FROM supplier_llm_v1 l
JOIN supplier_rules r USING (vendor_raw)
UNION ALL
SELECT DISTINCT vendor_raw, 'Private individual (withheld)', 'Individuals (withheld)', TRUE, NULL
FROM clean_spend
WHERE is_redacted
""")

con.sql("""
CREATE OR REPLACE TABLE dim_supplier AS
WITH per_name AS (
    SELECT
        supplier_name,
        arg_max(category_name, coalesce(total_spend, 0)) AS main_category,
        bool_or(is_individual)                          AS is_individual,
        COUNT(DISTINCT vendor_raw)                      AS raw_spellings
    FROM supplier_map
    GROUP BY supplier_name
)
SELECT ROW_NUMBER() OVER (ORDER BY supplier_name) AS supplier_key, *
FROM per_name
""")
con.sql("""
SELECT COUNT(*) AS suppliers,
       COUNT(*) FILTER (WHERE raw_spellings > 1) AS merged_from_several_spellings,
       MAX(raw_spellings) AS most_spellings
FROM dim_supplier
""").show()

┌───────────┬───────────────────────────────┬────────────────┐
│ suppliers │ merged_from_several_spellings │ most_spellings │
│   int64   │             int64             │     int64      │
├───────────┼───────────────────────────────┼────────────────┤
│      2332 │                           119 │             37 │
└───────────┴───────────────────────────────┴────────────────┘



## dim_service_area and dim_date
The date dimension includes the UK council financial year (April to March).

In [4]:
con.sql("""
CREATE OR REPLACE TABLE dim_service_area AS
SELECT ROW_NUMBER() OVER (ORDER BY service_area) AS service_area_key, service_area
FROM (SELECT DISTINCT service_area FROM clean_spend)
""")

con.sql("""
CREATE OR REPLACE TABLE dim_date AS
WITH bounds AS (SELECT MIN(posting_date) AS d0, MAX(posting_date) AS d1 FROM clean_spend),
days AS (SELECT CAST(range AS DATE) AS date FROM bounds, range(d0, d1 + INTERVAL 1 DAY, INTERVAL 1 DAY))
SELECT
    CAST(strftime(date, '%Y%m%d') AS INTEGER)        AS date_key,
    date,
    year(date)                                       AS year,
    month(date)                                      AS month,
    strftime(date, '%b')                             AS month_name,
    strftime(date, '%Y-%m')                          AS year_month,
    quarter(date)                                    AS calendar_quarter,
    CASE WHEN month(date) >= 4 THEN year(date) ELSE year(date) - 1 END AS fy_start_year,
    CASE WHEN month(date) >= 4
         THEN CAST(year(date) AS VARCHAR) || '/' || right(CAST(year(date) + 1 AS VARCHAR), 2)
         ELSE CAST(year(date) - 1 AS VARCHAR) || '/' || right(CAST(year(date) AS VARCHAR), 2)
    END                                              AS financial_year,
    strftime(date, '%A')                             AS day_name,
    isodow(date) >= 6                                AS is_weekend
FROM days
""")
con.sql("SELECT MIN(date), MAX(date), COUNT(*) AS days FROM dim_date").show()

┌────────────┬────────────┬───────┐
│ min(date)  │ max(date)  │ days  │
│    date    │    date    │ int64 │
├────────────┼────────────┼───────┤
│ 2023-11-21 │ 2025-06-30 │   588 │
└────────────┴────────────┴───────┘



## fact_spend
One row per payment.

In [5]:
con.sql("""
CREATE OR REPLACE TABLE fact_spend AS
SELECT
    ROW_NUMBER() OVER (ORDER BY s.posting_date, s.sap_doc_number)  AS payment_key,
    CAST(strftime(s.posting_date, '%Y%m%d') AS INTEGER)             AS date_key,
    ds.supplier_key,
    dc.category_key,
    sa.service_area_key,
    s.sap_doc_number,
    s.expense_type,
    s.file_month                                                    AS published_month,
    s.amount,
    s.amount < 0                                                    AS is_refund
FROM clean_spend s
JOIN supplier_map m     ON m.vendor_raw = s.vendor_raw
JOIN dim_supplier ds    ON ds.supplier_name = m.supplier_name
JOIN dim_category dc    ON dc.category_name = ds.main_category
JOIN dim_service_area sa ON sa.service_area = s.service_area
""")

CountBIGINT
77033


## Reconciliation
Row counts and totals must match the source exactly.

In [6]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM clean_spend)                         AS source_rows,
    (SELECT COUNT(*) FROM fact_spend)                          AS fact_rows,
    (SELECT SUM(amount) FROM clean_spend)                      AS source_total,
    (SELECT SUM(amount) FROM fact_spend)                       AS fact_total,
    (SELECT COUNT(*) FROM fact_spend f LEFT JOIN dim_date d USING (date_key) WHERE d.date_key IS NULL) AS missing_dates,
    (SELECT COUNT(*) FROM (SELECT vendor_raw FROM supplier_map GROUP BY 1 HAVING COUNT(*) > 1))       AS duplicate_mappings
""").show()

┌─────────────┬───────────┬───────────────┬───────────────┬───────────────┬────────────────────┐
│ source_rows │ fact_rows │ source_total  │  fact_total   │ missing_dates │ duplicate_mappings │
│    int64    │   int64   │ decimal(38,2) │ decimal(38,2) │     int64     │       int64        │
├─────────────┼───────────┼───────────────┼───────────────┼───────────────┼────────────────────┤
│       77033 │     77033 │  421596634.99 │  421596634.99 │             0 │                  0 │
└─────────────┴───────────┴───────────────┴───────────────┴───────────────┴────────────────────┘



## Sanity check: spend by category and top suppliers

In [7]:
con.sql("""
SELECT dc.category_name, ROUND(SUM(f.amount) / 1e6, 1) AS spend_m, COUNT(*) AS payments
FROM fact_spend f JOIN dim_category dc USING (category_key)
GROUP BY 1 ORDER BY spend_m DESC
""").show()

con.sql("""
SELECT ds.supplier_name, dc.category_name, ROUND(SUM(f.amount) / 1e6, 2) AS spend_m
FROM fact_spend f
JOIN dim_supplier ds USING (supplier_key)
JOIN dim_category dc USING (category_key)
WHERE NOT ds.is_individual
GROUP BY 1, 2 ORDER BY spend_m DESC LIMIT 10
""").show()

┌───────────────────────────┬─────────┬──────────┐
│       category_name       │ spend_m │ payments │
│          varchar          │ double  │  int64   │
├───────────────────────────┼─────────┼──────────┤
│ Social Care               │    99.0 │    33973 │
│ Education & Childcare     │    43.4 │     4820 │
│ Professional Services     │    42.8 │     2851 │
│ Public Sector Bodies      │    40.6 │      431 │
│ Waste & Environment       │    37.8 │      284 │
│ Transport & Travel        │    29.5 │     7073 │
│ Construction & Property   │    27.1 │     1853 │
│ Community & Voluntary     │    21.3 │     2766 │
│ Staffing & Recruitment    │    19.2 │     9783 │
│ Individuals (withheld)    │    16.4 │     8921 │
│ Housing & Accommodation   │    12.5 │     1382 │
│ IT & Telecoms             │    12.4 │      683 │
│ Energy & Utilities        │     9.9 │      419 │
│ Culture, Leisure & Events │     6.6 │      427 │
│ Goods & Supplies          │     2.8 │     1256 │
│ Other                     │  

## Export for Power BI

In [8]:
os.makedirs("../data/model", exist_ok=True)
for t in ["fact_spend", "dim_supplier", "dim_category", "dim_date", "dim_service_area"]:
    con.sql(f"COPY {t} TO '../data/model/{t}.csv' (HEADER)")
    print("exported", t, con.sql(f"SELECT COUNT(*) FROM {t}").fetchall()[0][0], "rows")
con.close()

exported fact_spend 77033 rows
exported dim_supplier 2332 rows
exported dim_category 16 rows
exported dim_date 588 rows
exported dim_service_area 16 rows
